# SpatialPeeler benchmark — Slide-seq Puck 06

Runs SpatialPeeler on the generated benchmark data from `benchmark.ipynb`.

**Setup:** Puck 06 bottom half split into case / control.  
Case = same tissue, but expression of the top 100 PC2-loaded genes is doubled (Poisson-perturbed) inside a central circle (~10% of beads).  
Control = the unmodified bottom section.

**Ground truth:** `obs['in_circle']` — beads that were perturbed in the case sample.

**Goal:** verify that SpatialPeeler recovers a factor whose p_hat score is elevated specifically inside the perturbed circle.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scanpy as sc
import anndata as ad
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score, roc_curve

warnings.simplefilter('ignore', category=ConvergenceWarning)
sc.settings.verbosity = 1

# SpatialPeeler imports
ROOT = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler')
sys.path.insert(0, str(ROOT))
from SpatialPeeler import case_prediction as cpred
from SpatialPeeler import plotting as plot

RAND_SEED = 28
np.random.seed(RAND_SEED)

## 1. Load benchmark data

In [ ]:
DATA_DIR = ROOT / 'benchmark' / 'generated_benchmark_data'

adata_ctrl = ad.read_h5ad(DATA_DIR/'adata06_top.h5ad')
adata_case = ad.read_h5ad(DATA_DIR/'adata06_bot_case.h5ad')

print('Control:', adata_ctrl)
print('Case:   ', adata_case)

In [ ]:
# Assign case/control labels
adata_ctrl.obs['sample_id'] = 'puck06_bot_ctrl'
adata_ctrl.obs['status']    = 0
adata_ctrl.obs['Condition'] = 'Control'

adata_case.obs['sample_id'] = 'puck06_bot_case'
adata_case.obs['status']    = 1
adata_case.obs['Condition'] = 'Case'


# Confirm ground truth column is present in both
print(f"'in_circle' in control obs: {'in_circle' in adata_ctrl.obs.columns}")
print(f"'in_circle' in case obs: {'in_circle' in adata_case.obs.columns}")

adata_ctrl.obs['in_circle'] = False
print(f"Control in_circle: {adata_ctrl.obs['in_circle'].sum():,} / {adata_ctrl.n_obs:,}")
print(f"Case in_circle: {adata_case.obs['in_circle'].sum():,} / {adata_case.n_obs:,}")

In [ ]:
print(adata_case.obs['in_circle'].head())
print(adata_ctrl.obs['in_circle'].head())

In [ ]:
top100_pc2 = pd.read_csv(DATA_DIR / 'top100_pc2_genes.csv')
sim_gene_symbols = top100_pc2['gene'].tolist()
print(f'Loaded {len(sim_gene_symbols)} simulation genes: {sim_gene_symbols[:8]} ...')

## 2. Concatenate and preprocess

In [ ]:
adata = ad.concat(
    [adata_ctrl, adata_case],
    join='inner',
    merge='first',     # preserve var columns (e.g. 'features') from first adata
    label='sample_id',
    keys=['puck06_bot_ctrl', 'puck06_bot_case'],
    index_unique='-'
)

for col in ['status', 'Condition', 'in_circle']:
    adata.obs[col] = np.concatenate([
        adata_ctrl.obs[col].values,
        adata_case.obs[col].values
    ])

adata.obs['status']    = adata.obs['status'].astype(int)
adata.obs['in_circle'] = adata.obs['in_circle'].astype(bool)

print(adata)
print('var columns:', adata.var.columns.tolist())
print(adata.obs[['sample_id', 'status', 'in_circle']].value_counts())

In [ ]:
adata.layers['counts'] = adata.X.copy()
print(f'Before spot filter: {adata.shape}')
# Custom gene filter: always keep sim genes regardless of min_cells
sim_set   = set(sim_gene_symbols)
is_sim    = np.array([g in sim_set for g in adata.var['features'].values])
min_cells = max(1, adata.n_obs // 500)
n_expr    = np.array((adata.X > 0).sum(axis=0)).flatten()
keep_genes = (n_expr >= min_cells) | is_sim
adata = adata[:, keep_genes].copy()
print(f'Gene filter: {keep_genes.sum():,} kept (min_cells={min_cells})')
print(f'Sim genes after gene filter: {np.array([g in sim_set for g in adata.var["features"].values]).sum()} / {len(sim_gene_symbols)}')

adata.obs['total_counts'] = np.array(adata.X.sum(axis=1)).flatten()
## no filtering of spots for now, since we want to keep all spots for benchmarking
#sc.pp.filter_cells(adata, min_counts=100)
print(f'After spot filter: {adata.shape}')

In [ ]:
present = set(adata.var['features'].values)
missing_sim_genes = [g for g in sim_gene_symbols if g not in present]
if missing_sim_genes:
    print(f"Warning: {len(missing_sim_genes)} sim genes missing: {missing_sim_genes}")
else:
    print(f"All {len(sim_gene_symbols)} simulation genes present after HVG subset.")


In [ ]:
adata.layers['lognorm'] = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4, layer='lognorm')
sc.pp.log1p(adata, layer='lognorm')

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    batch_key='sample_id',
    flavor='seurat',
    layer='lognorm',
    subset=False
)

# Force sim genes into HVG set using boolean mask
sim_set = set(sim_gene_symbols)
is_sim  = np.array([g in sim_set for g in adata.var['features'].values])
adata.var['highly_variable'] = adata.var['highly_variable'] | is_sim
print(f'HVG + sim: {adata.var["highly_variable"].sum()} genes')

adata = adata[:, adata.var['highly_variable']].copy()
sim_kept = np.array([g in sim_set for g in adata.var['features'].values]).sum()
print(f'After HVG subset: {adata.shape},  sim genes: {sim_kept}/{len(sim_gene_symbols)}')

In [ ]:
scale_features = True
if scale_features:
    # unit-variance scaling WITHOUT centering 
    # Keeps non-negativity (important for NMF) while matching the "variance normalize" idea.
    sc.pp.scale(adata, zero_center=False)

## 3. NMF decomposition

In [ ]:
N_FACTORS = 30

# Use unit-variance scaled data (adata.X after sc.pp.scale)
X_nmf = adata.X
if sp.issparse(X_nmf):
    X_nmf = X_nmf.toarray()
X_nmf = X_nmf.astype(np.float32)

nmf_model = NMF(
    n_components=N_FACTORS,
    init='nndsvda',
    random_state=RAND_SEED,
    max_iter=1000,
    solver='cd',
)
W = nmf_model.fit_transform(X_nmf)
H = nmf_model.components_

adata.obsm['X_nmf'] = W
adata.uns['nmf_components'] = H

print(f'NMF W: {W.shape},  reconstruction error: {nmf_model.reconstruction_err_:.2f}')

In [ ]:
sample_ids = adata.obs['sample_id'].unique().tolist()
adata_by_sample = {
    sid: adata[adata.obs['sample_id'] == sid].copy()
    for sid in sample_ids
}

for factor_idx in range(N_FACTORS): #range(N_FACTORS)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, sid in zip(axes, sample_ids):
        sub    = adata_by_sample[sid]
        x      = sub.obs['Raw_Slideseq_X'].values.astype(float)
        y      = sub.obs['Raw_Slideseq_Y'].values.astype(float)
        scores = sub.obsm['X_nmf'][:, factor_idx]
        sc_    = ax.scatter(x, y, c=scores, s=2, cmap='magma_r',
                            linewidths=0, rasterized=True)
        plt.colorbar(sc_, ax=ax, shrink=0.4, aspect=15, pad=0.02)
        ax.set_aspect('equal', 'box')
        ax.axis('off')
        ax.set_title(f'Factor {factor_idx+1} — {sid}', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
# Visualise sim gene expression: case vs control, side by side
GENES_TO_PLOT = sim_gene_symbols[:3]   # top 3 by |PC2 loading|

feat_to_idx = {g: i for i, g in enumerate(adata.var['features'].values)}

for gene in GENES_TO_PLOT:
    if gene not in feat_to_idx:
        print(f'{gene} not in adata after HVG subset — skipping')
        continue

    col = feat_to_idx[gene]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for ax, sid in zip(axes, sample_ids):
        sub  = adata[adata.obs['sample_id'] == sid]
        expr = sub.layers['counts'][:, col]
        if sp.issparse(expr):
            expr = expr.toarray().ravel()
        else:
            expr = np.asarray(expr).ravel()
        expr = np.log1p(expr)

        x    = sub.obs['Raw_Slideseq_X'].values.astype(float)
        y    = sub.obs['Raw_Slideseq_Y'].values.astype(float)

        sc_ = ax.scatter(x, y, c=expr, s=1, cmap='magma_r',linewidths=0, rasterized=True)
        plt.colorbar(sc_, ax=ax, fraction=0.03, pad=0.02, label='log1p(counts)')

        ax.set_aspect('equal', 'box'); ax.axis('off')
        ax.set_title(f'{gene}  —  {sid}', fontsize=12)

    plt.tight_layout()
    plt.show()

In [ ]:
nmf_df = pd.DataFrame(
    adata.obsm['X_nmf'],
    columns=[f'Factor {i+1}' for i in range(N_FACTORS)]
)
nmf_df['Condition'] = adata.obs['Condition'].values

nmf_long = nmf_df.melt(id_vars='Condition', var_name='Factor', value_name='Score')

fig, ax = plt.subplots(figsize=(N_FACTORS * 0.6, 5))
sns.violinplot(
    x='Factor', y='Score', hue='Condition', data=nmf_long,
    order=[f'Factor {i+1}' for i in range(N_FACTORS)],
    hue_order=['Control', 'Case'],
    palette={'Control': 'skyblue', 'Case': 'salmon'},
    inner='quartile', cut=0, density_norm='width',
    linewidth=0.6, ax=ax
)
ax.set_xlabel('')
ax.set_ylabel('NMF score', fontsize=11)
ax.set_title('NMF factor score distributions — Control vs Case', fontsize=12)
ax.legend(title='Condition', bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout()
plt.show()


## 4. Run SpatialPeeler

In [ ]:
results_v0 = cpred.single_factor_logistic_evaluation(
    adata,
    factor_key='X_nmf',
    max_factors=N_FACTORS
)

coef_df = pd.DataFrame({
    'factor':     [f'Factor_{r["factor_index"]+1}' for r in results_v0],
    'factor_idx': [r['factor_index'] for r in results_v0],
    'coef':       [r['coef'] for r in results_v0],
    'pval':       [r['pval'] for r in results_v0],
}).sort_values('coef', ascending=False)

print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(15, 6))
colors = ['#e84545' if c > 0 else '#4575b4' for c in coef_df['coef']]
ax.bar(coef_df['factor'], coef_df['coef'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Factor', fontsize=10)
ax.set_ylabel('Logistic regression coefficient', fontsize=14)
ax.set_title('SpatialPeeler: factor coefficients (case vs control)', fontsize=16)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import statsmodels.api as sm
from scipy.special import logit

X = adata.obsm['X_nmf']
y = adata.obs['status'].values        # 0 = Control, 1 = Case

THRESHOLD = 0.0    # exclude spots where NMF score == 0
VISUALIZE  = True  # set True for per-factor histograms and violin plots

all_results = []

for i in range(min(N_FACTORS, X.shape[1])):
    Xi        = X[:, i].reshape(-1, 1)
    high_mask = Xi.ravel() > THRESHOLD
    high_idx  = np.where(high_mask)[0]

    if len(high_idx) == 0:
        print(f'Factor {i+1}: no active spots — skipping.')
        continue
    if len(np.unique(y[high_idx])) < 2:
        print(f'Factor {i+1}: only one class in active spots — skipping.')
        continue

    Xi_high = Xi[high_idx]
    y_high  = y[high_idx]

    X_int = sm.add_constant(Xi_high)
    fit   = sm.Logit(y_high, X_int).fit(disp=False)
    p_hat = fit.predict(X_int)

    all_results.append({
        'factor_index': i,
        'coef':         float(fit.params[1]),
        'intercept':    float(fit.params[0]),
        'std_err':      float(fit.bse[1]),
        'pval':         float(fit.pvalues[1]),
        'p_hat':        p_hat,
        'logit_p_hat':  logit(p_hat),
        'high_idx':     high_idx,
        'y_high':       y_high,
    })

    if VISUALIZE:
        df_vis = pd.DataFrame({
            'Condition': adata.obs['Condition'].values[high_idx],
            'p_hat':     p_hat
        })
        from matplotlib.lines import Line2D
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        cond_colors = ['skyblue' if c == 0 else 'salmon' for c in y_high]
        axes[0].scatter(Xi_high.ravel(), p_hat, c=cond_colors,
                        s=3, alpha=0.4, linewidths=0, rasterized=True)
        axes[0].set_xlabel(f'NMF Factor {i+1} score', fontsize=9)
        axes[0].set_ylabel('p_hat', fontsize=9)
        axes[0].set_title(f'Factor {i+1} — NMF vs p_hat', fontsize=10)
        axes[0].legend(handles=[
            Line2D([0],[0], marker='o', color='w', markerfacecolor='skyblue', label='Control'),
            Line2D([0],[0], marker='o', color='w', markerfacecolor='salmon',  label='Case')],
            fontsize=8, frameon=False)
        sns.violinplot(x='Condition', y='p_hat', data=df_vis,
                       order=['Control', 'Case'],
                       palette={'Control': 'skyblue', 'Case': 'salmon'},
                       inner='box', cut=0, ax=axes[1])
        axes[1].set_ylim(0, 1)
        axes[1].set_title(f'Factor {i+1} — p_hat by condition', fontsize=10)
        plt.tight_layout(); plt.show()

results           = all_results
results_by_factor = {r['factor_index']: r for r in results}
print(f'Done. {len(results)} factors evaluated.')


In [ ]:

coef_df = pd.DataFrame({
    'factor':     [f'Factor_{r["factor_index"]+1}' for r in results],
    'factor_idx': [r['factor_index'] for r in results],
    'coef':       [r['coef'] for r in results],
    'pval':       [r['pval'] for r in results],
}).sort_values('coef', ascending=False)

print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(15, 5))
colors = ['#e84545' if c > 0 else '#4575b4' for c in coef_df['coef']]
ax.bar(coef_df['factor'], coef_df['coef'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Factor', fontsize=11)
ax.set_ylabel('Logistic regression coefficient', fontsize=11)
ax.set_title('SpatialPeeler (zero-filtered): factor coefficients', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.tight_layout(); plt.show()

## 5. Spatial visualisation of p_hat

In [ ]:
def _full_p_hat(r, n):
    """Reconstruct length-n p_hat; NaN for zero-NMF (inactive) spots."""
    ph = np.full(n, np.nan)
    ph[r['high_idx']] = r['p_hat']
    return ph

top_factors = coef_df[coef_df['coef'] > 0].head(5)['factor_idx'].tolist()
print('Top GOF factors:', [f+1 for f in top_factors])

for fi in top_factors:
    if fi not in results_by_factor:
        continue
    r    = results_by_factor[fi]
    ph   = _full_p_hat(r, adata.n_obs)   # NaN for inactive spots
    coef = coef_df.loc[coef_df['factor_idx'] == fi, 'coef'].values[0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, sid in zip(axes, sample_ids):
        sub      = adata[adata.obs['sample_id'] == sid]
        sub_idx  = np.where(adata.obs['sample_id'] == sid)[0]
        x        = sub.obs['Raw_Slideseq_X'].values.astype(float)
        y_coord  = sub.obs['Raw_Slideseq_Y'].values.astype(float)
        ph_sub   = ph[sub_idx]
        active   = ~np.isnan(ph_sub)

        # inactive spots in light grey
        ax.scatter(x[~active], y_coord[~active], c='#e8e8e8',
                   s=0.3, linewidths=0, rasterized=True)
        # active spots coloured by p_hat
        sc_ = ax.scatter(x[active], y_coord[active], c=ph_sub[active],
                         s=2, cmap='YlGnBu', linewidths=0, rasterized=True)
        plt.colorbar(sc_, ax=ax, shrink=0.6, aspect=15, pad=0.02, label='p_hat')

        ax.set_aspect('equal', 'box'); ax.axis('off')
        ax.set_title(f'Factor {fi+1} p_hat — {sid}  '
                     f'({active.sum():,} active spots)', fontsize=9)

    plt.suptitle(f'Factor {fi+1}  (coef={coef:.3f})', fontsize=11)
    plt.tight_layout(); plt.show()

## 6. Evaluate recovery of the perturbed circle

For each factor, compute AUROC between p_hat (on case beads only) and the ground-truth `in_circle` label.  
A high AUROC means the factor's p_hat distinguishes perturbed from unperturbed case beads.

In [ ]:
case_global_idx = np.where(adata.obs['status'].values == 1)[0]
gt = adata.obs['in_circle'].values[case_global_idx].astype(int)

def _case_p_hat(r, n, case_idx):
    """p_hat for case beads; 0 for inactive case beads (zero-NMF)."""
    ph = np.zeros(n)
    ph[r['high_idx']] = r['p_hat']
    return ph[case_idx]

auroc_rows = []
for r in results:
    ph_case = _case_p_hat(r, adata.n_obs, case_global_idx)
    try:
        auc = roc_auc_score(gt, ph_case) ## imported from sklearn.metrics 
    except Exception:
        auc = np.nan
    auroc_rows.append({
        'factor':     f'Factor_{r["factor_index"]+1}',
        'factor_idx': r['factor_index'],
        'coef':       r['coef'],
        'auroc':      auc
    })

auroc_df = pd.DataFrame(auroc_rows).sort_values('auroc', ascending=False)
print(auroc_df.to_string(index=False))

In [ ]:
# Bar chart of AUROC per factor
fig, ax = plt.subplots(figsize=(17, 4))
ax.bar(auroc_df['factor'], auroc_df['auroc'],
       color=['#e84545' if a >= 0.7 else '#aaaaaa' for a in auroc_df['auroc']])
ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--', label='random')
ax.set_ylim(0, 1)
ax.set_xlabel('Factor')
ax.set_ylabel('AUROC (p_hat vs in_circle, case beads only)')
ax.set_title('Circle recovery per factor')
ax.legend(fontsize=9)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
best_fi  = int(auroc_df.iloc[0]['factor_idx'])
best_auc = auroc_df.iloc[0]['auroc']
best_ph  = _case_p_hat(results_by_factor[best_fi], adata.n_obs, case_global_idx)

fpr, tpr, _ = roc_curve(gt, best_ph)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color='#e84545', lw=2,
        label=f'Factor {best_fi+1}  (AUC={best_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC — best factor (p_hat vs in_circle, case beads)')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
sub_case    = adata[adata.obs['sample_id'] == 'puck06_bot_case']
case_idx    = np.where(adata.obs['sample_id'] == 'puck06_bot_case')[0]
ph_full     = _full_p_hat(results_by_factor[best_fi], adata.n_obs)
ph_case     = ph_full[case_idx]
active_case = ~np.isnan(ph_case)
gt_bool     = sub_case.obs['in_circle'].values.astype(bool)
x  = sub_case.obs['Raw_Slideseq_X'].values.astype(float)
y  = sub_case.obs['Raw_Slideseq_Y'].values.astype(float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: p_hat (grey = inactive / zero-NMF)
axes[0].scatter(x[~active_case], y[~active_case], c='#e8e8e8',
                s=0.4, linewidths=0, rasterized=True)
sc_ = axes[0].scatter(x[active_case], y[active_case],
                      c=ph_case[active_case], s=0.6, cmap='magma_r',
                      vmin=0, vmax=1, linewidths=0, rasterized=True)
plt.colorbar(sc_, ax=axes[0], shrink=0.6, aspect=15, pad=0.02, label='p_hat')
axes[0].set_title(f'Case — Factor {best_fi+1} p_hat', fontsize=10)

# Right: ground truth
colors = np.where(gt_bool, '#e84545', '#d3d3d3')
axes[1].scatter(x, y, c=colors, s=0.5, linewidths=0, rasterized=True)
patches = [mpatches.Patch(color='#e84545', label='in circle (perturbed)'),
           mpatches.Patch(color='#d3d3d3', label='outside circle')]
axes[1].legend(handles=patches, fontsize=8, frameon=False, loc='lower right')
axes[1].set_title('Case — ground truth (in_circle)', fontsize=10)

for ax in axes:
    ax.set_aspect('equal', 'box'); ax.axis('off')

plt.suptitle(f'Best factor: Factor {best_fi+1}  (AUROC={best_auc:.3f})', fontsize=11)
plt.tight_layout(); plt.show()

## 7. Spot clustering and DE analysis

For each top GOF factor: cluster active spots into `case_0/1` (KMeans on case p_hat) and `control_0/1` (split by the same threshold), then run Wilcoxon DE between every pair of clusters.

In [ ]:
from sklearn.cluster import KMeans

CASE_COND_NAME  = 'Case'
FDR_THRESH      = 0.05
LOGFC_THRESH    = 0.1

# gene ID → symbol mapping for DE output
var_to_symbol = dict(zip(adata.var_names, adata.var['features']))

def get_num_sig_de(de_df, fdr=0.05, logfc=0.1):
    return ((de_df['pvals_adj'] < fdr) & (de_df['logfoldchanges'].abs() > logfc)).sum()

def run_wilcoxon(adata_sub, grp_mask, ref_mask, grp_label, ref_label):
    keep = grp_mask | ref_mask
    ad   = adata_sub[keep].copy()
    ad.obs['_grp'] = pd.Categorical(
        np.where(grp_mask[keep], grp_label, ref_label),
        categories=[ref_label, grp_label]
    )
    sc.tl.rank_genes_groups(
        ad, groupby='_grp', groups=[grp_label], reference=ref_label,
        method='wilcoxon', corr_method='benjamini-hochberg',
        use_raw=False, layer='lognorm', n_genes=ad.n_vars
    )
    de = sc.get.rank_genes_groups_df(ad, group=grp_label).rename(columns={'names': 'gene'})
    de['gene_name'] = de['gene'].map(var_to_symbol)
    return de

# ── factors to process: top GOF (positive coefficient) ───────────
top_gof_idx = coef_df[coef_df['coef'] > 0]['factor_idx'].tolist()[:3]
print(f'Processing {len(top_gof_idx)} GOF factors: {[i+1 for i in top_gof_idx]}')


cluster_mask_dict          = {}
de_results_dict_factorwise = {}

CLUSTER_PALETTE = {
    'control_0': '#54A24B',
    'control_1': '#E45756',
    'case_0':    '#4C78A8',
    'case_1':    '#F58518',
}
CAT_ORDER = ['control_0', 'control_1', 'case_0', 'case_1']

for factor_idx in top_gof_idx:
    if factor_idx not in results_by_factor:
        continue

    print(f"\n{'='*55}\nFactor {factor_idx+1}")
    result   = results_by_factor[factor_idx]
    high_idx = result['high_idx']
    p_hat    = result['p_hat']

    adata_sub = adata[high_idx].copy()
    adata_sub.obs['p_hat'] = p_hat

    # ── 1. KMeans on case p_hat → derive threshold ──────────────────────
    mask_case  = adata_sub.obs['Condition'].values == CASE_COND_NAME
    p_hat_case = p_hat[mask_case]

    km       = KMeans(n_clusters=2, random_state=RAND_SEED, n_init='auto')
    km.fit(p_hat_case.reshape(-1, 1))
    centers  = km.cluster_centers_.ravel()
    order    = np.argsort(centers)           # [low_label, high_label]
    remap    = {order[0]: 0, order[1]: 1}
    case_km  = np.vectorize(remap.get)(km.labels_)  # 0=low, 1=high phat
    threshold = centers.mean()
    print(f"  KMeans centroids: {centers.round(3)},  threshold: {threshold:.3f}")

    # ── 2. Build cluster labels for all active spots ─────────────────────
    obs_col       = f'cluster_f{factor_idx+1}'
    cluster_labels = np.empty(len(adata_sub), dtype=object)

    case_local = np.where(mask_case)[0]
    ctrl_local = np.where(~mask_case)[0]
    cluster_labels[case_local] = np.where(case_km == 1, 'case_1', 'case_0')
    cluster_labels[ctrl_local] = np.where(p_hat[~mask_case] >= threshold, 'control_1', 'control_0')

    adata_sub.obs[obs_col] = pd.Categorical(cluster_labels, categories=CAT_ORDER)
    print(adata_sub.obs[obs_col].value_counts().reindex(CAT_ORDER).to_string())
    cluster_mask_dict[obs_col] = adata_sub.obs[obs_col].copy()

    # ── 3. Violin: p_hat per cluster ─────────────────────────────────────
    df_vio = pd.DataFrame({'cluster': adata_sub.obs[obs_col].values, 'p_hat': p_hat})
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.violinplot(x='cluster', y='p_hat', data=df_vio, order=CAT_ORDER,
                   palette=CLUSTER_PALETTE, inner='box', cut=0, ax=ax)
    ax.set_ylim(0, 1)
    ax.set_xlabel('')
    ax.set_title(f'Factor {factor_idx+1} — p_hat by cluster', fontsize=11)
    plt.tight_layout(); plt.show()

    # ── 4. Spatial: cluster labels, case and control panels ───────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, sid in zip(axes, sample_ids):
        sub    = adata_sub[adata_sub.obs['sample_id'] == sid]
        x      = sub.obs['Raw_Slideseq_X'].values.astype(float)
        y_coord= sub.obs['Raw_Slideseq_Y'].values.astype(float)
        colors = [CLUSTER_PALETTE[c] for c in sub.obs[obs_col]]
        ax.scatter(x, y_coord, c=colors, s=0.5, linewidths=0, rasterized=True)
        ax.set_aspect('equal', 'box'); ax.axis('off')
        ax.set_title(f'Factor {factor_idx+1} — {sid}', fontsize=9)
    handles = [mpatches.Patch(color=CLUSTER_PALETTE[k], label=k) for k in CAT_ORDER]
    axes[1].legend(handles=handles, fontsize=8, frameon=False, loc='lower right')
    plt.tight_layout(); plt.show()

    # ── 5. Wilcoxon DE for each comparison ───────────────────────────────
    obs       = adata_sub.obs[obs_col].values
    m         = {k: obs == k for k in CAT_ORDER}

    comparisons = [
        ('case1_vs_case0',       m['case_1'],    m['case_0']),
        ('case1_vs_other',       m['case_1'],    ~m['case_1']),
        ('case1_vs_control1',    m['case_1'],    m['control_1']),
        ('case0_vs_control',     m['case_0'],    m['control_0'] | m['control_1']),
        ('case0_vs_control0',    m['case_0'],    m['control_0']),
        ('control1_vs_control0', m['control_1'], m['control_0']),
    ]

    de_results_dict = {}
    num_sig_DE      = {}

    for comp_name, grp_mask, ref_mask in comparisons:
        grp_label, ref_label = comp_name.split('_vs_')
        if grp_mask.sum() < 5 or ref_mask.sum() < 5:
            print(f"  {comp_name}: too few spots — skipped")
            continue
        de = run_wilcoxon(adata_sub, grp_mask, ref_mask, grp_label, ref_label)
        n  = get_num_sig_de(de, FDR_THRESH, LOGFC_THRESH)
        print(f"  {comp_name}: {n} sig DE genes")
        de_results_dict[comp_name] = de
        num_sig_DE[comp_name]      = n

    de_results_dict_factorwise[f'factor_{factor_idx+1}'] = de_results_dict

    # ── 6. Bar chart: sig DE genes per comparison ─────────────────────────
    if num_sig_DE:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(list(num_sig_DE.keys()), list(num_sig_DE.values()),
               color=sns.color_palette('viridis', len(num_sig_DE)))
        ax.set_ylabel('# sig DE genes')
        ax.set_title(f'Factor {factor_idx+1} — significant DE genes per comparison', fontsize=11)
        plt.xticks(rotation=40, ha='right', fontsize=9)
        plt.tight_layout(); plt.show()

print('\nDone.')

What the perturbation does:
pc2_loadings.abs().nlargest(100) selects genes by absolute loading magnitude. Looking at the actual values: 97 out of 100 genes have NEGATIVE PC2 loadings. Only Camk1d (+0.581), Lars2 (+0.175), and Gm48099 (+0.033) are positive.

The perturbation then adds Poisson(λ) counts to all 100 genes, where λ = mean expression in the circle. This means:

For Camk1d (loading +0.581, λ=1.17): adding counts pushes circle beads toward positive PC2
For the 97 negatively-loaded genes (Ywhab −0.204, Rbfox1 −0.192, Syn2 −0.182, Mbp, Plp1…): adding counts pushes circle beads toward negative PC2
In NMF space: those 97 genes are broad neuronal/glial markers (Syn2, Rbfox1, Mbp, Plp1, Mobp…) that are co-expressed throughout cortical tissue — including the control (top puck). They will form NMF factors that are already high in control. Boosting them inside the circle makes those spots look more control-like, not more case-specific. That's directly why Factor 30 gets a negative beta — the circle has elevated expression of a factor that the logistic regression associates with control.
The design issue: using .abs() selects genes from both ends of PC2. You're simultaneously boosting the "Camk1d direction" (3 genes) and the "everything else direction" (97 genes). The signal goes in opposite directions at once — no coherent new NMF factor emerges.

The fix would be to perturb only positively-loaded PC2 genes, or select genes that consistently separate case from control (e.g., by DE between top/bottom halves before perturbation).

Factor 1 has AUROC=0.856 and coef=167.079 — an enormous coefficient. That tells you Factor 1 has near-zero scores everywhere except the case circle beads.

Here's why k matters:

With k=30: NMF has 30 slots to cover the variation across ~80k spots and 2025 genes. The dominant co-expression signals are the broad neuronal/glial patterns (Ywhab, Rbfox1, Syn2, Mbp, Plp1, Mobp…) — those 97 negatively-loaded genes are all highly correlated across brain tissue and eat up most of the factor budget. Camk1d's signal (the strongest single driver of the perturbation at λ=1.17) gets subsumed into a broader factor. There's no dedicated slot left to isolate the incremental Camk1d signal that's specific to the circle. The "closest" factor (Factor 30) ends up dominated by those broadly-expressed neuronal genes, which are also high in control → negative beta.

With k=50: NMF has enough components to separately represent (1) baseline Camk1d expression throughout the tissue, and (2) the extra Camk1d signal concentrated in the case circle — that delta becomes its own dedicated factor. Because Camk1d is doubled in the circle (ctrl=1.17, case=2.35) and is essentially not doubled anywhere in control, that incremental component is case-circle-specific. The logistic regression then sees: high Factor 1 → almost exclusively case circle → coefficient explodes to 167.

In general: NMF with low k conflates spatially distinct signals into shared factors. Higher k allows finer-grained decomposition that can isolate spatially restricted perturbations. The practical implication for your benchmark is that the signal is recoverable, but k=30 was too coarse to separate the Camk1d-driven circle signal from the dominant neuronal background.